# check how many discontinuties there are in the new flights
# see how many are pia-> ladd/ ladd-> pia vs just airport small diffrence.
# Maybe this should be a score thing or it can be absolute. 

In [3]:
from datetime import date

import polars as pl
from airports.airportLookup import AirportLookup
from data_engineering.flights.flight_type import get_flights
airport_lookup = AirportLookup()
df = get_flights(date(2026,6,2), date(2026,6,8))

In [5]:
df = df.filter(pl.col("ladd") | pl.col("pia")).with_columns(
    pl.col("takeoff_airport_ident")
    .map_elements(airport_lookup.airport_ident_is_in_us_faa_airspace, return_dtype=pl.Boolean)
    .alias("takeoff_airport_in_us_faa_airspace"),
    pl.col("landing_airport_ident")
    .map_elements(airport_lookup.airport_ident_is_in_us_faa_airspace, return_dtype=pl.Boolean)
    .alias("landing_airport_in_us_faa_airspace"),
)

In [6]:
df.filter("takeoff_airport_in_us_faa_airspace")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_in_us_faa_airspace,landing_airport_in_us_faa_airspace
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str,bool,bool
"""0a8040""","""GGT60""","""C6-ZIP""",2026-06-02 20:08:54.110,"""KBTV""",null,2026-06-02 22:37:48.330,"""MYAX""",null,null,null,null,null,null,null,false,true,false,false,"""LJ60""",null,"""LEARJET 60""","""A2""",true,false
"""0a80a1""","""REE210""","""C6-DON""",2026-06-02 14:34:42.740,"""KORL""",null,2026-06-02 15:37:56.120,"""MYGW""",null,null,null,null,null,null,null,false,true,false,false,"""C208""",null,"""CESSNA 208 Caravan""","""A1""",true,false
"""0c414f""","""HI1025""","""HI1025""",2026-06-02 13:25:54.870,"""KTEB""",null,2026-06-02 22:38:16.810,"""KFRG""",null,null,null,null,null,null,null,false,true,false,false,"""GLF4""",null,"""GULFSTREAM 4""","""A3""",true,true
"""0c4153""","""HI1045""","""HI1045""",2026-06-02 13:54:17.860,"""KFXE""",null,2026-06-02 16:28:20.820,"""TUPA""",null,null,null,null,null,null,null,false,true,false,false,"""C25B""",null,"""CESSNA 525B Citation CJ3""","""A1""",true,false
"""0d0665""","""00000000""","""XA-UPL""",2026-06-02 14:51:01.040,"""KONT""",null,2026-06-02 15:25:54.480,"""MMTJ""",null,null,null,null,null,null,null,false,true,false,false,"""CVLT""",null,"""CANADAIR Cosmopolitan""","""A0""",true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""c07f7c""","""CGWHF""","""C-GWHF""",2026-06-07 15:11:06.260,"""KBED""",null,2026-06-07 16:18:02.790,"""CYYZ""",null,null,null,null,null,null,null,false,true,false,false,"""GLEX""","""Skyservice Business Aviation I…","""BOMBARDIER BD-700 Global 6000/…","""A3""",true,false
"""c0866a""","""PAG808""","""C-GYXL""",2026-06-07 21:31:20.830,"""Y98""",null,2026-06-07 21:37:54.890,"""US-0951""",null,null,null,null,null,null,null,false,true,false,false,"""SW4""","""PERIMETER AVIATION LP AS REPRE…","""SWEARINGEN Metro""","""A2""",true,true
"""c0866a""","""PAG808""","""C-GYXL""",2026-06-07 22:17:40.150,"""5Y0""",null,2026-06-07 22:57:25.680,"""CYFD""",null,null,null,null,null,null,null,false,true,false,false,"""SW4""","""PERIMETER AVIATION LP AS REPRE…","""SWEARINGEN Metro""","""A2""",true,false


In [7]:
route_mismatches = (
    df
    .sort(["icao", "takeoff_time"])
    .with_columns(
        pl.col("landing_airport_ident").shift(1).over("icao").alias("prev_landing_airport_ident"),
        pl.col("landing_time").shift(1).over("icao").alias("prev_landing_time"),
    )
    .filter(
        pl.col("prev_landing_airport_ident").is_not_null()
        & (pl.col("prev_landing_airport_ident") != pl.col("takeoff_airport_ident"))
    )
)

route_mismatches = route_mismatches.filter(pl.col("takeoff_airport_in_us_faa_airspace") & pl.col("landing_airport_in_us_faa_airspace"))
from airports.airportLookup import AirportLookup, haversine

airport_lookup = AirportLookup()

def airport_distance_km(row):
    prev_landing = airport_lookup.get_Airport_from_airport_ident(
        row["prev_landing_airport_ident"]
    )
    next_takeoff = airport_lookup.get_Airport_from_airport_ident(
        row["takeoff_airport_ident"]
    )

    if prev_landing is None or next_takeoff is None:
        return None

    return haversine(
        prev_landing.lat,
        prev_landing.lon,
        next_takeoff.lat,
        next_takeoff.lon,
    )

route_mismatches = route_mismatches.with_columns(
    pl.struct(["prev_landing_airport_ident", "takeoff_airport_ident"])
    .map_elements(airport_distance_km, return_dtype=pl.Float64)
    .alias("airport_disagreement_distance_km")
)

In [16]:
from data_engineering.adsb.read_adsb import is_american_icao


route_mismatches.filter(is_american_icao())

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_in_us_faa_airspace,landing_airport_in_us_faa_airspace,prev_landing_airport_ident,prev_landing_time,airport_disagreement_distance_km
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str,bool,bool,str,datetime[ms],f64
"""a0000d""","""N1AL""","""N1AL""",2026-06-02 20:00:24.550,"""KBZN""",null,2026-06-02 22:47:01.100,"""KCAK""",null,null,null,null,null,null,null,false,true,false,false,"""GLF6""","""ALC 6091 LLC""","""GULFSTREAM G650""","""A3""",true,true,"""05MT""",2026-06-02 17:01:14.780,4.062056
"""a0001a""","""CXF1""","""N1AZ""",2026-06-03 01:22:10.570,"""69OK""",null,2026-06-03 03:29:35.250,"""KSDL""",null,null,null,null,null,null,null,false,true,false,false,"""FA7X""","""WILMINGTON TRUST CO TRUSTEE""","""DASSAULT Falcon 7X""","""A2""",true,true,"""KGMJ""",2026-06-03 00:52:24.850,15.757328
"""a00033""","""N1BZ""","""N1BZ""",2026-06-06 22:00:51.690,"""KMVN""",null,2026-06-06 23:28:40.300,"""KRNH""",null,null,null,null,null,null,null,false,true,false,false,"""C25A""","""L2 OPERATIONS LLC""","""CESSNA 525A Citation CJ2""","""A1""",true,true,"""KHSB""",2026-06-06 21:18:32.360,62.977102
"""a00082""","""N1FC""","""N1FC""",2026-06-03 21:40:23.230,"""KLAX""",null,2026-06-04 00:18:48.860,"""KAUS""",null,null,null,null,null,null,null,false,true,false,false,"""G280""","""TVPX ARS INC TRUSTEE""","""GULFSTREAM G280""","""A2""",true,true,"""KHHR""",2026-06-03 17:44:45.770,7.082036
"""a00082""","""N1FC""","""N1FC""",2026-06-06 17:15:54.620,"""KERV""",null,2026-06-06 17:34:27.570,"""KAUS""",null,null,null,null,null,null,null,false,true,false,false,"""G280""","""TVPX ARS INC TRUSTEE""","""GULFSTREAM G280""","""A2""",true,true,"""US-1199""",2026-06-06 15:17:58.670,26.411615
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""adf2dc""",null,"""N9981F""",2026-06-03 16:46:28.540,"""7TX5""",null,2026-06-03 16:53:41.710,"""K63F""",null,null,null,null,null,null,null,false,true,false,false,"""C72R""","""KCSI AERIAL PATROL INC""","""CESSNA 172R Cutlass RG""",null,true,true,"""7TX7""",2026-06-03 15:14:11.840,20.777855
"""adf2dc""",null,"""N9981F""",2026-06-03 17:13:29.990,"""KMDD""",null,2026-06-03 17:24:29.670,"""KMDD""",null,null,null,null,null,null,null,false,true,false,false,"""C72R""","""KCSI AERIAL PATROL INC""","""CESSNA 172R Cutlass RG""",null,true,true,"""K63F""",2026-06-03 16:53:41.710,30.380816
"""adf2dc""",null,"""N9981F""",2026-06-03 18:14:29.020,"""KMAF""",null,2026-06-03 19:35:56.590,"""7TX5""",null,null,null,null,null,null,null,false,true,false,false,"""C72R""","""KCSI AERIAL PATROL INC""","""CESSNA 172R Cutlass RG""",null,true,true,"""KMDD""",2026-06-03 17:24:29.670,14.141805


In [20]:
df.filter(pl.col("icao") == "a00639")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_in_us_faa_airspace,landing_airport_in_us_faa_airspace
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str,bool,bool
"""a00639""","""N100RP""","""N100RP""",2026-06-02 20:07:37.360,"""KNUQ""",null,2026-06-03 01:05:48.620,"""KOPF""",null,null,null,null,null,null,null,false,true,false,false,"""C172""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""CESSNA 172 Skyhawk""","""A3""",true,true
"""a00639""","""N111MF""","""N100RP""",2026-06-03 01:28:48.660,"""KAJO""",null,2026-06-03 02:01:55.920,"""KVNY""",null,null,null,null,null,null,null,false,true,false,false,"""C172""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""CESSNA 172 Skyhawk""","""A1""",true,true
"""a00639""","""N111MF""","""N100RP""",2026-06-07 20:19:42.150,"""0CA3""",null,2026-06-07 20:45:35.150,"""L52""",null,null,null,null,null,null,null,false,true,false,false,"""C172""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""CESSNA 172 Skyhawk""","""A1""",true,true
"""a00639""","""N111MF""","""N100RP""",2026-06-07 23:39:30.710,"""L52""",null,2026-06-07 23:47:14.210,"""L52""",null,null,null,null,null,null,null,false,true,false,false,"""C172""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""CESSNA 172 Skyhawk""","""A1""",true,true


In [19]:
route_mismatches.filter(is_american_icao()).filter(pl.col("airport_disagreement_distance_km") > 300)

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_in_us_faa_airspace,landing_airport_in_us_faa_airspace,prev_landing_airport_ident,prev_landing_time,airport_disagreement_distance_km
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str,bool,bool,str,datetime[ms],f64
"""a00639""","""N111MF""","""N100RP""",2026-06-03 01:28:48.660,"""KAJO""",null,2026-06-03 02:01:55.920,"""KVNY""",null,null,null,null,null,null,null,false,true,false,false,"""C172""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""CESSNA 172 Skyhawk""","""A1""",true,true,"""KOPF""",2026-06-03 01:05:48.620,3684.260779
"""a00965""","""N101L""","""N101L""",2026-06-07 22:38:06.030,"""KT45""",null,2026-06-07 23:20:38.050,"""KPWA""",null,null,null,null,null,null,null,false,true,false,false,"""PC12""","""JTL HOLDINGS LLC, C/O THOMAS L…","""PILATUS PC-12""","""A1""",true,true,"""NM63""",2026-06-07 21:53:45.410,422.428983
"""a00e2d""","""N102WY""","""N102WY""",2026-06-02 22:54:08.800,"""K9U4""",null,2026-06-02 23:18:52.550,"""KCYS""",null,null,null,null,null,null,null,false,true,false,false,"""C560""","""WYOMING DEPARTMENT OF TRANSPOR…","""CESSNA 560 Citation Ultra""","""A2""",true,true,"""WY30""",2026-06-02 14:43:27.390,337.39205
"""a021e9""","""N1076K""","""N1076K""",2026-06-06 04:09:08.680,"""KFLG""",null,2026-06-06 04:18:30.580,"""KFLG""",null,null,null,null,null,null,null,false,true,false,false,"""BE9L""","""GUARDIAN FLIGHT LLC""","""BEECH 90 King Air""","""A1""",true,true,"""US-5300""",2026-06-04 23:29:15.360,473.609279
"""a0224c""","""N1079D""","""N1079D""",2026-06-05 02:00:11.610,"""US-1539""",null,2026-06-05 02:10:03.350,"""KPVU""",null,null,null,null,null,null,null,false,true,false,false,"""BE9L""","""GUARDIAN FLIGHT LLC""","""BEECH 90 King Air""","""A1""",true,true,"""US-5355""",2026-06-04 11:32:31.970,326.319218
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ad9470""","""OME10""","""N974VV""",2026-06-07 15:04:30.600,"""US-2818""",null,2026-06-07 16:12:51.790,"""KVCV""",null,null,null,null,null,null,null,false,true,false,false,"""DC10""","""AIRCRAFT ENDEAVORS LLC""","""MCDONNELL DOUGLAS DC-10""","""A5""",true,true,"""AU-0498""",2026-06-03 00:51:34.170,12168.758527
"""add61d""","""N9908K""","""N9908K""",2026-06-05 00:01:10.320,"""3OK7""",null,2026-06-05 00:10:35.590,"""KRVS""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true,"""KBJC""",2026-06-04 17:47:21.500,885.115617
"""add61d""","""N9908K""","""N9908K""",2026-06-05 13:04:25.110,"""KBJC""",null,2026-06-05 13:17:38.150,"""KBJC""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true,"""KH76""",2026-06-05 02:09:09.610,840.671505


In [21]:
df.filter(pl.col("icao") == "add61d")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_in_us_faa_airspace,landing_airport_in_us_faa_airspace
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str,bool,bool
"""add61d""","""N9908K""","""N9908K""",2026-06-02 15:39:32.610,"""KLMO""",null,2026-06-02 15:51:01.610,"""CD39""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true
"""add61d""","""N9908K""","""N9908K""",2026-06-02 16:46:12.930,"""0CD7""",null,2026-06-02 16:53:48.820,"""KBJC""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true
"""add61d""","""N9908K""","""N9908K""",2026-06-03 16:36:47.250,"""CO58""",null,2026-06-03 16:46:03.190,"""KEIK""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true
"""add61d""","""N9908K""","""N9908K""",2026-06-03 17:04:12.650,"""CO58""",null,2026-06-03 17:13:22.860,"""KLMO""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true
"""add61d""","""N9908K""","""N9908K""",2026-06-03 17:37:44.360,"""US-0074""",null,2026-06-03 17:43:01.870,"""KBJC""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""add61d""","""N9908K""","""N9908K""",2026-06-05 19:31:16.930,"""KEIK""",null,2026-06-05 19:36:45.180,"""7CO0""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true
"""add61d""","""N9908K""","""N9908K""",2026-06-05 19:38:36.180,"""US-6650""",null,2026-06-05 19:45:55.160,"""KBJC""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,true
"""add61d""","""N9908K""","""N9908K""",2026-06-05 23:27:03.730,"""KEIK""",null,2026-06-05 23:32:32.770,"""US-8441""",null,null,null,null,null,null,null,false,true,false,false,"""P28A""","""G&M AIRCRAFT INC""","""PIPER PA-28-140/150/160/180""","""A1""",true,false


In [29]:
from data_engineering.adsb.read_adsb import read_adsb
from datetime import date, datetime

df_adsb = read_adsb(date(2026,3,1))
df_adsb = df_adsb.filter(pl.col("icao") == "a921a9")

In [30]:
df_adsb.filter(pl.col("time") > datetime(2026,3,1,21,40))

registration,ladd,baro_altitude_ft,category,on_ground,callsign,military,pia,owner,icao,interesting,lat,ground_speed_kt,track_deg,aircraft_description,aircraft_type,time,lon
str,bool,i32,str,bool,str,bool,bool,str,str,bool,f64,f32,f32,str,str,datetime[ms],f64
"""N688HL""",true,225,null,false,null,false,false,"""H&L 168 LLC""","""a921a9""",false,35.238831,147.0,303.899994,"""GULFSTREAM G200""","""GALX""",2026-03-01 21:45:09.750,-120.64373
"""N688HL""",true,325,null,false,null,false,false,"""H&L 168 LLC""","""a921a9""",false,35.240112,152.800003,303.799988,"""GULFSTREAM G200""","""GALX""",2026-03-01 21:45:12.910,-120.646019
"""N688HL""",true,325,null,false,null,false,false,"""H&L 168 LLC""","""a921a9""",false,35.240341,155.0,303.700012,"""GULFSTREAM G200""","""GALX""",2026-03-01 21:45:13.580,-120.646477
"""N688HL""",true,400,"""A2""",false,"""N688HL""",false,false,"""H&L 168 LLC""","""a921a9""",false,35.240936,156.399994,303.799988,"""GULFSTREAM G200""","""GALX""",2026-03-01 21:45:15.080,-120.647564
"""N688HL""",true,400,null,false,null,false,false,"""H&L 168 LLC""","""a921a9""",false,35.241165,160.600006,304.100006,"""GULFSTREAM G200""","""GALX""",2026-03-01 21:45:15.730,-120.647964
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""N688HL""",true,21425,null,false,null,false,false,"""H&L 168 LLC""","""a921a9""",false,43.459763,412.0,39.5,"""GULFSTREAM G200""","""GALX""",2026-03-01 23:11:23.390,-110.535555
"""N688HL""",true,21375,null,false,null,false,false,"""H&L 168 LLC""","""a921a9""",false,43.460632,412.0,39.5,"""GULFSTREAM G200""","""GALX""",2026-03-01 23:11:24.090,-110.534597
"""N688HL""",true,21250,null,false,null,false,false,"""H&L 168 LLC""","""a921a9""",false,43.464288,411.299988,39.599998,"""GULFSTREAM G200""","""GALX""",2026-03-01 23:11:26.580,-110.530439


In [33]:
df.filter(pl.col("icao") == "a921a9")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,takeoff_airport_in_us_faa_airspace,landing_airport_in_us_faa_airspace
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,bool,bool,bool,bool,str,str,str,str,bool,bool
"""a921a9""","""""","""""",2026-03-01 18:58:35.550,"""KONT""",9.866336,2026-03-01 19:39:49.740,"""KSBP""",1.284632,false,true,false,false,"""""","""""","""""","""""",true,true
"""a921a9""","""""","""""",2026-03-01 23:56:26.210,"""KJAC""",7.70013,2026-03-02 01:47:29.600,"""KMEV""",0.978543,false,true,false,false,"""""","""""","""""","""""",true,true
